# Comment Category Prediction

## Submission By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

## 1. Import all the required libraries

In [ ]:
import importlib

if not importlib.util.find_spec("catboost"):
    print("Installing catboost...")
    %pip install catboost

In [ ]:
import gc
import os
import re
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from IPython.display import display
from scipy.optimize import minimize
from scipy.sparse import csr_matrix, hstack, issparse
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    MaxAbsScaler,
    OrdinalEncoder,
    PowerTransformer,
    RobustScaler,
)
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

## 2. Configuration

In [ ]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("datasets/")

TARGET_COL = "label"
TEXT_COL = "comment"
ID_COL = "ID"
TEST_SIZE = 0.2

# Ensemble top K models
TOP_K = 3

# Random State for reproducibility
RANDOM_STATE = 42

# Controls HPT
PERFORM_TUNING = False

# Cross-validation splits
CV_SPLITS = 3

# Controls XG Boost (can be slow to train)
ENABLE_XGBOOST = False

# Dimensionality reduction for text features
ENABLE_SVD = False
SVD_N_COMPONENTS = 500

# Controls GPU acceleration for gradient boosting models
ENABLE_GPU = False
GPU_DEVICE_ID = "0"

## 3. Helper Functions

In [ ]:
def header(msg: str) -> None:
    """
    Print a formatted header message.
    :param msg: Message to be printed as header.
    :return:
    """
    print("\n" + "=" * 50)
    print(msg.center(50))
    print("=" * 50)

In [ ]:
def summarize_frame(name: str, frame: pd.DataFrame) -> None:
    """Print a lightweight audit of a dataframe."""
    header(f"{name} summary")
    print(f"Shape: {frame.shape}")
    print("\nColumn types:")
    print(frame.dtypes.sort_index())
    print("\nMissing values (top 10):")
    print(frame.isna().sum().sort_values(ascending=False).head(10))
    display(frame.head())

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean the input text by removing URLs, special characters, and extra spaces.
    :param text: Input text string.
    :return: Cleaned text string.
    """
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def calc_sparsity(matrix) -> float:
    """
    Calculate the sparsity of a given matrix.
    :param matrix: Input matrix (numpy array or sparse matrix).
    :return: Sparsity in percentage.
    """
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero_elements = matrix.nnz if issparse(matrix) else np.count_nonzero(matrix)
    sparsity = 1.0 - (non_zero_elements / total_elements)
    return sparsity * 100

In [ ]:
def build_leaderboard(results: dict) -> pd.DataFrame:
    """Convert the model-results dictionary into a sorted leaderboard."""
    if not results:
        return pd.DataFrame(columns=["model_name", "macro_f1"])

    leaderboard = pd.DataFrame(
        [
            {"model_name": model_name, "macro_f1": data["score"]}
            for model_name, data in results.items()
        ]
    )
    return leaderboard.sort_values("macro_f1", ascending=False, ignore_index=True)

## 4. Data loading

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

In [ ]:
for name, frame in {
    "train": train,
    "test": test,
    "sample submission": sample_sub,
}.items():
    summarize_frame(name, frame)

## 5. Exploratory Data Analysis

In [ ]:
# Drop rows with missing comments, as they cannot be used for modeling
train.dropna(subset=["comment"], inplace=True)

In [ ]:
header("Dataset shapes")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Sample submission shape: {sample_sub.shape}")

In [ ]:
eda_train = train.assign(
    comment_length=train[TEXT_COL].fillna("").str.len(),
    word_count=train[TEXT_COL].fillna("").str.split().str.len(),
)

header("Sample training rows")
eda_train.sample(min(5, len(eda_train)), random_state=RANDOM_STATE)

In [ ]:
# Label distribution
print(train["label"].value_counts())
print(f"\nProportions:\n{train['label'].value_counts(normalize=True)}")

In [ ]:
train.describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(data=eda_train, x=TARGET_COL, ax=axes[0])
axes[0].set_title("Label distribution")

numeric_corr = eda_train.select_dtypes(include="number").corr(numeric_only=True)
sns.heatmap(numeric_corr, cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Correlation matrix")

plt.tight_layout()
plt.show()

In [ ]:
train["comment_length"] = train["comment"].str.len()
train["word_count"] = train["comment"].str.split().str.len()

header("TEXT STATISTICS BY LABEL")
print(train.groupby("label")[["comment_length", "word_count"]].mean())

In [ ]:
cat_features = train.select_dtypes(include="object" if IS_KAGGLE else "str").columns
cat_features = [col for col in cat_features if col not in ["comment", "created_date"]]

header("Categorical Features")
for col in cat_features:
    print(f"\n{col} Value counts:")
    print(train[col].value_counts())
    print(f"\n{col} Proportions:")
    print(train[col].value_counts(normalize=True))
    print("\n" + "-" * 50)

In [ ]:
header("Boxplots of Numerical Features to Identify Outliers")

num_cols = train.select_dtypes(include="number").columns.tolist()

for i in range(0, len(num_cols), 4):
    batch = num_cols[i : i + 4]
    fig, axes = plt.subplots(1, len(batch), figsize=(16, 4))
    axes = [axes] if len(batch) == 1 else axes

    for ax, col in zip(axes, batch):
        sns.boxplot(y=train[col], ax=ax, showfliers=False)
        sns.stripplot(y=train[col], ax=ax, alpha=0.3, size=2, color="orange")
        ax.set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
header("Numerical Feature Distributions")

num_cols = train.select_dtypes(include="number").columns.tolist()

for i in range(0, len(num_cols), 4):
    batch = num_cols[i : i + 4]
    fig, axes = plt.subplots(1, len(batch), figsize=(16, 4))
    axes = [axes] if len(batch) == 1 else axes

    for ax, col in zip(axes, batch):
        sns.histplot(train[col], ax=ax, kde=True, bins=30)
        ax.set_title(col)

    plt.tight_layout()
    plt.show()

## 6. Feature Engineering

In [ ]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """Create a wide set of text, metadata, and interaction features."""
    df = df.copy()

    # Imputation
    for col in ["race", "religion", "gender"]:
        df[col] = df[col].fillna("none").astype(str).str.lower()

    # Text features
    df["comment_length"] = df["comment"].str.len()
    df["word_count"] = df["comment"].str.split().str.len()
    df["avg_word_length"] = df["comment"].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
    )
    df["unique_word_count"] = df["comment"].apply(
        lambda x: len(set(str(x).lower().split()))
    )
    df["unique_word_ratio"] = df["unique_word_count"] / (df["word_count"] + 1)

    # Special characters, punctuation and intensity
    df["exclamation_count"] = df["comment"].str.count("!")
    df["question_count"] = df["comment"].str.count(r"\?")
    df["capital_ratio"] = df["comment"].apply(
        lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
    )
    df["digit_count"] = df["comment"].str.count(r"\d")
    df["special_char_count"] = df["comment"].str.count(r"[^a-zA-Z0-9\s]")
    df["uppercase_word_count"] = df["comment"].apply(
        lambda x: sum(1 for w in str(x).split() if w.isupper() and len(w) > 1)
    )
    df["punctuation_ratio"] = df["special_char_count"] / (df["comment_length"] + 1)
    df["repeat_exclaim"] = df["comment"].str.count(r"!{2,}")
    df["repeat_question"] = df["comment"].str.count(r"\?{2,}")
    df["elongated_word_count"] = df["comment"].str.count(r"\b\w*(\w)\1{2,}\w*\b")

    # Lightweight lexicon features
    profanity_terms = [
        "idiot",
        "stupid",
        "dumb",
        "hate",
        "kill",
        "moron",
        "trash",
        "rubish",
    ]
    identity_terms = [
        "black",
        "white",
        "asian",
        "latino",
        "hindu",
        "muslim",
        "christian",
        "jewish",
        "female",
        "male",
        "transgender",
        "lgbtq",
    ]
    profanity_pattern = r"\b(?:" + "|".join(profanity_terms) + r")\b"
    identity_pattern = r"\b(?:" + "|".join(identity_terms) + r")\b"
    df["profanity_count"] = df["comment"].str.lower().str.count(profanity_pattern)
    df["identity_term_count"] = df["comment"].str.lower().str.count(identity_pattern)
    df["has_profanity"] = (df["profanity_count"] > 0).astype(int)

    # Engagement features
    df["total_votes"] = df["upvote"] + df["downvote"]
    df["vote_ratio"] = np.where(
        df["total_votes"] > 0, df["upvote"] / df["total_votes"], 0.5
    )
    df["vote_diff"] = df["upvote"] - df["downvote"]
    df["log_upvote"] = np.log1p(df["upvote"])
    df["log_downvote"] = np.log1p(df["downvote"])
    df["vote_product"] = np.log1p(df["upvote"] * df["downvote"])
    df["controversy_score"] = np.where(
        df["total_votes"] > 0,
        1 - np.abs(df["upvote"] - df["downvote"]) / df["total_votes"],
        0,
    )

    # Length-normalized engagement
    df["downvote_per_word"] = df["downvote"] / (df["word_count"] + 1)
    df["upvote_per_word"] = df["upvote"] / (df["word_count"] + 1)
    df["vote_diff_per_100_chars"] = 100 * df["vote_diff"] / (df["comment_length"] + 1)

    # Temporal features
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["hour"] = df["created_date"].dt.hour
    df["day_of_week"] = df["created_date"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 6)).astype(int)
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    # Emoticon features
    df["total_emoticons"] = df["emoticon_1"] + df["emoticon_2"] + df["emoticon_3"]
    df["has_emoticon"] = (df["total_emoticons"] > 0).astype(int)
    df["emoticon_ratio"] = df["total_emoticons"] / (df["word_count"] + 1)

    # Convert boolean features to integers
    bool_features = df.select_dtypes(include="bool").columns
    df[bool_features] = df[bool_features].astype(int)

    # Demographics + pairwise interactions
    for col in ["race", "religion", "gender"]:
        vc = df[col].value_counts(normalize=True)
        rare_values = vc[vc < 0.01].index
        df[col] = df[col].where(~df[col].isin(rare_values), "other_rare")

    df["race_x_religion"] = df["race"] + "_" + df["religion"]
    df["race_x_gender"] = df["race"] + "_" + df["gender"]
    df["religion_x_gender"] = df["religion"] + "_" + df["gender"]
    df["demographics"] = df["race"] + "_" + df["religion"] + "_" + df["gender"]

    # Indicator combinations
    df["indicator_sum"] = (
        (df["race"] != "none").astype(int)
        + (df["religion"] != "none").astype(int)
        + (df["gender"] != "none").astype(int)
        + df["disability"]
    )
    df["if_sum"] = df["if_1"] + df["if_2"]
    df["if_product"] = df["if_1"] * df["if_2"]

    # Tag metadata as pseudo-text
    df["metadata_text"] = (
        "race_"
        + df["race"].astype(str)
        + " religion_"  # type: ignore[ty:unsupported-operator]
        + df["religion"].astype(str)
        + " gender_"
        + df["gender"].astype(str)
        + " disability_"
        + df["disability"].astype(str)
        + " if1_"
        + df["if_1"].astype(str)
        + " if2_"
        + df["if_2"].astype(str)
    )

    # Rank signals
    df["upvote_vs_post"] = df["upvote"] - df["post_avg_upvote"]
    df["downvote_vs_post"] = df["downvote"] - df["post_avg_downvote"]
    df["upvote_zscore"] = df["upvote_vs_post"] / (df["post_std_upvote"] + 1e-8)
    df["downvote_zscore"] = df["downvote_vs_post"] / (df["post_std_downvote"] + 1e-8)

    df["upvote_rank_pct"] = df.groupby("post_id")["upvote"].rank(pct=True)
    df["downvote_rank_pct"] = df.groupby("post_id")["downvote"].rank(pct=True)
    df["vote_diff_rank_pct"] = df.groupby("post_id")["vote_diff"].rank(pct=True)
    df["is_high_activity_post"] = (df["post_count"] >= 8).astype(int)

    # Text x engagement interactions
    df["length_x_downvote"] = df["comment_length"] * df["log_downvote"]
    df["words_x_downvote"] = df["word_count"] * df["log_downvote"]

    # Keep features numerically stable
    for col in ["vote_diff_per_100_chars", "upvote_zscore", "downvote_zscore"]:
        df[col] = np.clip(df[col], -50, 50)

    # Drop noisy features
    df.drop(columns=["created_date", "post_id"], inplace=True)

    return df

In [ ]:
def add_target_encoding(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cols: list[str],
    target: str,
    n_splits: int = 5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add out-of-fold target-encoding features for selected categorical columns."""
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    train_df = train_df.copy()
    test_df = test_df.copy()

    classes = sorted(train_df[target].unique())
    for col in cols:
        for cls in classes:
            col_name = f"{col}_te_{cls}"
            train_df[col_name] = 0.0
            te_test = np.zeros(len(test_df))
            for fold, (tr_idx, val_idx) in enumerate(
                kf.split(train_df, train_df[target])
            ):
                binary_target = (train_df.iloc[tr_idx][target] == cls).astype(float)

                means = binary_target.groupby(train_df.iloc[tr_idx][col]).mean()
                global_mean = binary_target.mean()

                train_df.iloc[val_idx, train_df.columns.get_loc(col_name)] = (
                    train_df.iloc[val_idx][col].map(means).fillna(global_mean)
                )
                te_test += test_df[col].map(means).fillna(global_mean).values  # type: ignore[ty:unsupported-operator]

            test_df[col_name] = te_test / n_splits

    return train_df, test_df

In [ ]:
train, test = add_target_encoding(
    train,
    test,
    ["race", "religion", "gender"],
    "label",
)

In [ ]:
post_stats = (
    train.groupby("post_id")
    .agg(
        post_avg_upvote=("upvote", "mean"),
        post_count=("upvote", "count"),
        post_avg_downvote=("downvote", "mean"),
        post_std_upvote=("upvote", "std"),
        post_std_downvote=("downvote", "std"),
        post_label_mean=("label", "mean"),
        post_label_std=("label", "std"),
    )
    .reset_index()
    .fillna(0)
)

# Merge post-level statistics computed from the training split
train = train.merge(post_stats, on="post_id", how="left")
test = test.merge(post_stats, on="post_id", how="left")

In [ ]:
train = create_features(train)
test = create_features(test)

In [ ]:
header("Numerical Feature Distributions After Feature Engineering")

num_cols = train.select_dtypes(include="number").columns.tolist()

for i in range(0, len(num_cols), 4):
    batch = num_cols[i : i + 4]
    fig, axes = plt.subplots(1, len(batch), figsize=(16, 4))
    axes = [axes] if len(batch) == 1 else axes

    for ax, col in zip(axes, batch):
        sns.histplot(train[col], ax=ax, kde=True, bins=30)
        ax.set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
cat_features = train.select_dtypes(
    include=("object" if IS_KAGGLE else "str")
).columns.tolist()
cat_features.remove("comment")
cat_features

## 7. Preprocessing and Feature Assembly

The feature pipeline uses three branches:

1. ordinal-encoded categorical metadata,
2. scaled numeric features,
3. sparse TF-IDF text vectors built from cleaned comments.

In [ ]:
cat_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value="none")),
        (
            "onehot",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

transformer = ColumnTransformer(
    [("cat_process", cat_pipeline, cat_features)],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

# Separate label before transformation (test doesn't have label column)
y_labels = train["label"].copy()
train.drop(columns=["label"], inplace=True)

train = transformer.fit_transform(train)
test = transformer.transform(test)

# Add label back to train
train["label"] = y_labels.values

In [ ]:
header("Post-Preprocessing Dataset Info")
print(f"Encoded train shape: {train.shape}")
print(f"Encoded test shape: {test.shape}")
print(f"Total Features: {len(train.columns)}")
print(f"Encoded categorical columns: {cat_features}")
print(f"Output Columns: {train.columns.tolist()}")

In [ ]:
# Clean the raw comment text before vectorization.
train["comment_clean"] = train["comment"].apply(clean_text)
test["comment_clean"] = test["comment"].apply(clean_text)

In [ ]:
# Word-level TF-IDF (Higher max_features + trigrams)
tfidf_word = TfidfVectorizer(
    max_features=50_000,
    ngram_range=(1, 2),
    min_df=15,
    max_df=0.90,
    sublinear_tf=True,
    dtype=np.float32,
    stop_words="english",
)

# Character-level TF-IDF (Captures subword patterns/misspellings)
tfidf_char = TfidfVectorizer(
    max_features=40_000,
    ngram_range=(3, 4),
    min_df=15,
    max_df=0.90,
    sublinear_tf=True,
    analyzer="char_wb",
    dtype=np.float32,
)

In [ ]:
tfidf_word_train = tfidf_word.fit_transform(train["comment_clean"])
tfidf_word_test = tfidf_word.transform(test["comment_clean"])

tfidf_char_train = tfidf_char.fit_transform(train["comment_clean"])
tfidf_char_test = tfidf_char.transform(test["comment_clean"])

print(f"Word TF-IDF: {tfidf_word_train.shape}")
print(f"Char TF-IDF: {tfidf_char_train.shape}")
gc.collect()

In [ ]:
train.drop(columns=["comment", "comment_clean"], inplace=True)
test.drop(columns=["comment", "comment_clean"], inplace=True)

In [ ]:
X_text_train = hstack([tfidf_word_train, tfidf_char_train]).tocsr()
X_text_test = hstack([tfidf_word_test, tfidf_char_test]).tocsr()

if ENABLE_SVD:
    header("Applying TruncatedSVD on text features")
    svd = TruncatedSVD(
        n_components=SVD_N_COMPONENTS,
        n_iter=7,
        random_state=RANDOM_STATE,
    )
    X_text_train = csr_matrix(svd.fit_transform(X_text_train).astype(np.float32))
    X_text_test = csr_matrix(svd.transform(X_text_test).astype(np.float32))
    print(f"SVD explained variance ratio: {svd.explained_variance_ratio_.sum():.4f}")
    print(f"SVD text shape (train): {X_text_train.shape}")
else:
    print("SVD disabled; using raw sparse TF-IDF text features.")

In [ ]:
feature_cols = train.select_dtypes(include="number").columns.tolist()

# remove label from features
feature_cols.remove("label")
num_cols = [c for c in feature_cols if c not in cat_features]

In [ ]:
train[num_cols] = train[num_cols].astype("float32")
test[num_cols] = test[num_cols].astype("float32")
train[cat_features] = train[cat_features].astype("int32")
test[cat_features] = test[cat_features].astype("int32")

In [ ]:
X_tab_num = train[num_cols].fillna(0)
X_tab_num_test = test[num_cols].fillna(0)
X_tab_cat = train[cat_features].fillna(-1)
X_tab_cat_test = test[cat_features].fillna(-1)

In [ ]:
# separate columns with enough variance for PowerTransformer
low_variance_cols = [c for c in X_tab_num.columns if X_tab_num[c].nunique() <= 5]
high_variance_cols = [c for c in X_tab_num.columns if c not in low_variance_cols]

if high_variance_cols:
    # clip extreme values
    clip_cols = high_variance_cols
    X_tab_num[clip_cols] = X_tab_num[clip_cols].clip(
        lower=X_tab_num[clip_cols].quantile(0.001),
        upper=X_tab_num[clip_cols].quantile(0.999),
        axis=1,
    )

    # apply PowerTransformer to handle skewness only on high variance columns
    pt = PowerTransformer(method="yeo-johnson")
    X_tab_num[high_variance_cols] = pt.fit_transform(X_tab_num[high_variance_cols])
    X_tab_num_test[high_variance_cols] = pt.transform(
        X_tab_num_test[high_variance_cols]
    )

# scale all numeric features
scaler = RobustScaler().set_output(transform="pandas")
X_tab_num_scaled = scaler.fit_transform(X_tab_num)
X_tab_num_test_scaled = scaler.transform(X_tab_num_test)

In [ ]:
header("Numerical Feature Distributions After Scaling")

num_cols = X_tab_num_scaled.select_dtypes(include="number").columns.tolist()

for i in range(0, len(num_cols), 4):
    batch = num_cols[i : i + 4]
    fig, axes = plt.subplots(1, len(batch), figsize=(16, 4))
    axes = [axes] if len(batch) == 1 else axes

    for ax, col in zip(axes, batch):
        sns.histplot(X_tab_num_scaled[col], ax=ax, kde=True, bins=30)
        ax.set_title(col)

    plt.tight_layout()
    plt.show()

In [ ]:
# Combine all features: text branch (raw TF-IDF or SVD) + numeric + categorical
X = hstack([X_text_train, csr_matrix(X_tab_num_scaled), csr_matrix(X_tab_cat)]).tocsr()
X_sub = hstack(
    [X_text_test, csr_matrix(X_tab_num_test_scaled), csr_matrix(X_tab_cat_test)]
).tocsr()
y = train["label"].values

cat_indices = list(range(X.shape[1] - len(cat_features), X.shape[1]))

print(f"Combined training shape: {X.shape}")
print(f"Combined test shape: {X_sub.shape}")
gc.collect()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
header("Sparse Matrix Stats")

print("Train Matrix:")
print(f"\tShape: {X.shape}")
print(f"\tSparsity: {calc_sparsity(X):.2f}%")
print(f"\tSparse size: {X.data.nbytes / 1e6:.2f} MB")

print("\nSubmission Matrix:")
print(f"\tShape: {X_sub.shape}")
print(f"\tSparsity: {calc_sparsity(X_sub):.2f}%")
print(f"\tSparse size: {X_sub.data.nbytes / 1e6:.2f} MB")

# Cleanup
gc.collect()

## Model Building

In [ ]:
xgboost_device_kwargs = (
    {"device": "cuda", "max_bin": 64} if ENABLE_GPU else {"device": "cpu"}
)

lightgbm_device_kwargs = (
    {"device_type": "cuda", "gpu_device_id": int(GPU_DEVICE_ID), "max_bin": 64}
    if ENABLE_GPU
    else {"device_type": "cpu"}
)

catboost_device_kwargs = (
    {"task_type": "GPU", "devices": GPU_DEVICE_ID}
    if ENABLE_GPU
    else {"task_type": "CPU"}
)

In [ ]:
models = {
    # Gradient Boosting Models
    "XGBoost": XGBClassifier(
        n_estimators=1500,
        max_depth=6,
        learning_rate=0.02,
        subsample=0.8,
        colsample_bytree=0.6,
        min_child_weight=3,
        random_state=RANDOM_STATE,
        objective="multi:softprob",
        num_class=4,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        # early_stopping_rounds=50,
        **xgboost_device_kwargs,
    ),
    "LightGBM": lgb.LGBMClassifier(
        n_estimators=1500,
        max_depth=-1,
        num_leaves=127,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=30,
        objective="multiclass",
        random_state=RANDOM_STATE,
        verbose=-1,
        class_weight="balanced",
        n_jobs=-1,
        **lightgbm_device_kwargs,  # type: ignore[ty:invalid-argument-type]
    ),
    # "CatBoost": CatBoostClassifier(
    #     loss_function="MultiClass",
    #     random_state=RANDOM_STATE,
    #     **catboost_device_kwargs,
    # ),
    # "HistGradientBoosting": HistGradientBoostingClassifier(
    #     max_iter=1000,
    #     learning_rate=0.05,
    #     max_depth=8,
    #     l2_regularization=0.1,
    #     class_weight="balanced",
    #     early_stopping=True,
    #     random_state=RANDOM_STATE,
    # ),
    # Tree-Based Ensemble Models
    # "RandomForest": RandomForestClassifier(
    #     n_estimators=300,
    #     max_depth=10,
    #     random_state=RANDOM_STATE,
    #     class_weight="balanced",
    #     n_jobs=-1,
    # ),
    # "ExtraTrees": ExtraTreesClassifier(
    #     n_estimators=300,
    #     max_depth=10,
    #     random_state=RANDOM_STATE,
    #     class_weight="balanced",
    #     n_jobs=-1,
    # ),
    # Linear Models
    # "LogisticRegression": LogisticRegression(
    #     C=10.0,
    #     max_iter=2000,
    #     random_state=RANDOM_STATE,
    #     class_weight="balanced",
    #     solver="saga",
    #     n_jobs=-1,
    # ),
    "LinearSVC_Calibrated": CalibratedClassifierCV(
        estimator=LinearSVC(
            C=1.0,
            dual=False,
            class_weight=None,
            max_iter=3000,
            loss="squared_hinge",
            random_state=RANDOM_STATE,
        ),
        method="sigmoid",
        cv=2,
    ),
    # Neural Network
    "MLP": MLPClassifier(
        hidden_layer_sizes=(256, 128, 64),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        batch_size=256,
        learning_rate="adaptive",
        max_iter=500,
        random_state=RANDOM_STATE,
        early_stopping=True,
    ),
}

In [ ]:
scores = {}

for model_name, model in models.items():
    header(f"Training {model_name}")
    if not ENABLE_XGBOOST and "XGBoost" in model_name:
        print(f"Skipping {model_name} due to long training time.\n")
        continue

    if model_name == "XGBoost":
        sample_weights = compute_sample_weight("balanced", y_train)
        model.fit(
            X_train,
            y_train,
            sample_weight=sample_weights,
        )
    elif model_name == "LightGBM":
        model.fit(
            X_train,
            y_train,
            categorical_feature=cat_indices,  # type: ignore[ty:unknown-argument]
        )
    else:
        model.fit(X_train, y_train)

    val_preds = model.predict(X_test)

    score = f1_score(y_test, val_preds, average="macro")
    scores[model_name] = {
        "model": model,
        "score": score,
    }

    print(f"\t{model_name} Macro F1 Score: {score:.4f}")

In [ ]:
# sort scores
scores = list(sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True))

In [ ]:
header("Validation leaderboard")
build_leaderboard(dict(scores))

## 9. Hyperparameter Tuning

In [ ]:
params_dict = {
    # Gradient Boosting Models
    "XGBoost": {
        "n_estimators": [200, 300, 500],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_child_weight": [1, 5, 10],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
    },
    "LightGBM": {
        "n_estimators": [200, 300, 500],
        "num_leaves": [31, 63, 127],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_child_samples": [10, 20, 50],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "class_weight": ["balanced", None],
    },
    "HistGradientBoosting": {
        "max_iter": [200, 300, 500],
        "max_depth": [5, 7, 10, None],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_samples_leaf": [10, 20, 50],
        "l2_regularization": [0, 0.1, 1],
        "class_weight": ["balanced", None],
    },
    # Tree-Based Ensemble Models
    "RandomForest": {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.3],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    "ExtraTrees": {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.3],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    # Linear Models
    "LogisticRegression": {
        "C": [0.01, 0.1, 1, 10],
        "class_weight": ["balanced", None],
    },
    # Neural Network
    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64), (128, 64, 32)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.01],
        "batch_size": [64, 128, 256],
    },
}

In [ ]:
if PERFORM_TUNING:
    for model_name, model_data in scores[:3]:
        header(f"Hyperparameter Tuning for {model_name}")

        fit_params = {}
        if model_name == "XGBoost":
            weights = compute_sample_weight("balanced", y_train)
            fit_params["sample_weight"] = weights

        search_cv = RandomizedSearchCV(
            estimator=models[model_name],
            param_distributions=params_dict[model_name],
            n_iter=5,
            scoring="f1_macro",
            cv=3,
            verbose=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        search_cv.fit(X_train, y_train, **fit_params)
        scores.append(
            (
                f"{model_name}_Tuned",
                {
                    "score": search_cv.best_score_,
                    "model": search_cv.best_estimator_,
                },
            )
        )

        print(f"\t{model_name}_Tuned Best Score: {search_cv.best_score_:.4f}")
else:
    print("Skipping hyperparameter tuning (PERFORM_TUNING = False).")
    print("Using optimized default parameters instead.")

In [ ]:
# sort scores
scores = list(sorted(scores, key=lambda x: x[1]["score"], reverse=True))

In [ ]:
header("Validation leaderboard")
build_leaderboard(dict(scores))

## 10. Weighted Ensemble

In [ ]:
classes = np.unique(y)

n_models = min(len(scores), TOP_K)
model_preds = []
model_names = []
for model_name, model_data in scores[:n_models]:
    header(f"Ensemble Model: {model_name}")
    print(f"Macro F1 Score: {model_data['score']:.4f}")
    model = model_data["model"]
    model_preds.append(model.predict_proba(X_test))
    model_names.append(model_name)
    classes = model.classes_

model_preds = np.array(model_preds)

header("Finding optimal weights")


def objective_func(log_weights: np.ndarray) -> float:
    """Optimize weights in log-space so they always sum to one after softmax."""
    normalized_weights = np.exp(log_weights) / np.exp(log_weights).sum()
    blended_probs = np.tensordot(
        normalized_weights,
        model_preds,
        axes=(0, 0),
    )
    blended_preds = classes[np.argmax(blended_probs, axis=1)]
    return -f1_score(y_test, blended_preds, average="macro")


initial_weights = np.zeros(n_models)
result = minimize(objective_func, initial_weights, method="Nelder-Mead")
optimal_weights = np.exp(result.x) / np.sum(np.exp(result.x))

optimal_weights_dict = {name: w for name, w in zip(model_names, optimal_weights)}

print(f"\nOptimal weights found:")
for name, w in optimal_weights_dict.items():
    print(f"  {name}: {w:.4f}")

test_probs = np.tensordot(optimal_weights, model_preds, axes=(0, 0))

In [ ]:
header("Ensemble Model Results")
ensemble_pred_indices = np.argmax(test_probs, axis=1)
ensemble_preds = classes[ensemble_pred_indices]

ensemble_model_name = f"Ensemble_Top_{TOP_K}"
ensemble_score = f1_score(y_test, ensemble_preds, average="macro")

scores.append(
    (
        ensemble_model_name,
        {
            "score": ensemble_score,
            "model": None,
        },
    )
)

print(f"Ensemble Macro F1 Score: {ensemble_score:.4f}")

print("\nEnsemble Classification Report:")
print(classification_report(y_test, ensemble_preds))

## 11. Model Evaluation

In [ ]:
scores = list(sorted(scores, key=lambda x: x[1]["score"], reverse=True))

header("Final Leaderboard")
print(f"{'Rank':<5} {'Model Name':<25} {'Score':<10}")
print("-" * 65)

for rank, (name, data) in enumerate(scores, 1):
    print(f"{rank:<5} {name:<25} {data['score']:.4f}")

# Update best_model_data to the new #1
best_model_name, best_model_data = scores[0]
print(
    f"\n>> New Best Model: {best_model_name} with Score: {best_model_data['score']:.4f}"
)

## 12. Predicting Test Data

In [ ]:
if best_model_name == ensemble_model_name:
    # Use ensemble model
    final_probs = np.zeros((X_sub.shape[0], len(np.unique(y))))
    print("Generating Ensemble submission...")
    header("Retraining Top Models on Full Data")

    selected_models = [m for m in scores if m[0] in optimal_weights_dict]

    for model_name, model_data in selected_models:
        model_weight = optimal_weights_dict[model_name]
        if not ENABLE_XGBOOST and "XGBoost" in model_name:
            print(f"Skipping retraining for {model_name} due to long training time.\n")
            continue

        print(f"Retraining {model_name}...")
        print(f"Macro F1 Score: {model_data['score']:.4f}")

        print("Retraining on full training data...")
        model = model_data["model"]

        if "XGBoost" in model_name:
            sample_weights = compute_sample_weight("balanced", y)
            model.fit(X, y, sample_weight=sample_weights)
        elif "LightGBM" in model_name:
            model.fit(X, y, categorical_feature=cat_indices)
        else:
            model.fit(X, y)

        # Final predictions
        model_preds = model.predict_proba(X_sub)
        final_probs += model_preds * model_weight

        print(f"{model_name} retrained on full data...\n")
else:
    # Use single best model
    print(
        f"Ensemble score ({ensemble_score:.4f}) did not beat single best model ({best_model_data['score']:.4f})."
    )

    header(f"Best Model: {best_model_name}")
    print(f"Macro F1 Score: {best_model_data['score']:.4f}")

    print("Classification Report:")
    best_val_preds = best_model_data["model"].predict(X_test)
    print(classification_report(y_test, best_val_preds))

    print("\nRetraining on full training data...")
    model = best_model_data["model"]

    if "XGBoost" in best_model_name:
        sample_weights = compute_sample_weight("balanced", y)
        model.fit(X, y, sample_weight=sample_weights)
    elif "LightGBM" in best_model_name:
        model.fit(X, y, categorical_feature=cat_indices)
    else:
        model.fit(X, y)

    final_probs = model.predict_proba(X_sub)

In [ ]:
# Final predictions
final_pred_indices = np.argmax(final_probs, axis=1)
final_preds = classes[final_pred_indices]
print("Final submission generated.")

## 13. Final Submission

In [ ]:
submission = pd.DataFrame({"ID": test.index + 1, "label": final_preds})
submission.to_csv("submission.csv", index=False)

header("Submission created!")
print(submission["label"].value_counts())

--- 
**⚠️ Automated Merge Info**

The cells below were added automatically by the merge script to control execution flow.

In [ ]:
# stop execution here

import sys
sys.exit(0)

# Milestones

## Milestone 1: Data loading and preprocessing

**Deadline:** `Feb 15, 2026`

- Data loading
- Exploratory Data Analysis
    - Data visualization
    - Statistical analysis
- Train-Validation Split
- Data Cleaning/ Preprocessing
- Building a baseline model.

### Import libraries

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

### Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

### Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

### Question 1

What is the shape of the training dataset?

- [ ] (102000, 15)
- [ ] (198000, 14)
- [x] (198000, 15)
- [ ] (300000, 15)

In [4]:
train.shape

(198000, 15)

### Question 2

How many columns are present in the test dataset?

In [5]:
test.shape[1]

14

### Question 3

How many columns in the training dataset have **object** data type?

In [6]:
train.select_dtypes(include="object").shape[1]

C:\Users\varun\AppData\Local\Temp\ipykernel_9608\1487577859.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  train.select_dtypes(include="object").shape[1]


5

### Question 4

How many **numerical** columns are present in the training dataset?

In [7]:
train.select_dtypes(include="number").shape[1]

9

### Question 5

Which of the following columns is of boolean type?

- [ ] gender
- [ ] race
- [x] disability
- [ ] emoticon_1

In [8]:
train.select_dtypes(include="bool").columns

Index(['disability'], dtype='str')

### Question 6

Which of the following columns have missing values?

- [x] race
- [x] religion
- [x] gender
- [x] comment

In [9]:
train.columns[train.isna().sum() > 0]

Index(['race', 'religion', 'gender', 'comment'], dtype='str')

### Question 7

How many distinct target classes are present in the dataset?

In [10]:
train["label"].nunique()

4

### Question 8

What percentage of the dataset approximately belongs to label 0?

- [ ] ~30%
- [ ] ~45%
- [x] ~58%
- [ ] ~75%

In [11]:
np.round(train["label"].value_counts(normalize=True) * 100)  # type: ignore[ty:unsupported-operator]

label
0    58.0
2    32.0
1     8.0
3     3.0
Name: proportion, dtype: float64

### Question 9

What is the median number of upvotes per comment?

- [ ] 0
- [x] 1
- [ ] 2
- [ ] 3

In [12]:
train["upvote"].quantile(0.5)

np.float64(1.0)

### Question 10

Which numerical feature shows the largest maximum value?

- [ ] upvote
- [ ] downvote
- [x] if_1
- [ ] if_2

In [13]:
train.select_dtypes(include="number").max().sort_values(ascending=False)

if_1          1860
if_2          1833
upvote         201
post_id        129
downvote       107
emoticon_1      47
emoticon_3      17
emoticon_2      11
label            3
dtype: int64

### Question 11

What is the minimum value of `if_2`?

In [14]:
train["if_2"].min()

np.int64(3)

## Milestone 2: Model Training and Hyperparameter Tuning

**Deadline:** `Feb 22, 2026`

1. [Linear Models](https://scikit-learn.org/stable/modules/linear_model.html)
2. [Stochastic Gradient Descent](https://scikit-learn.org/stable/modules/sgd.html)

Tune the hyperparameters of the model.

### Import libraries

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

### Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

### Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

### Question 1

In the context of text preprocessing for NLP, which condition must be checked in addition to NaN values when assessing missingness in the comment column of train.csv?

- [ ] Presence of duplicate comments
- [ ] Presence of out-of-vocabulary tokens
- [x] Presence of empty or whitespace-only strings
- [ ] Presence of rare words

### Question 2

Convert the created_date column into datetime objects. Identify the month that occurs the most number of times in the dataset and enter the answer as a lowercase string.

In [4]:
train["created_date"] = pd.to_datetime(train["created_date"])
most_common_month = train["created_date"].dt.month.mode()[0]
pd.to_datetime(most_common_month, format="%m").strftime("%B").lower()

'may'

### Question 3

Create a new feature called total_emoticons by calculating the sum of `emoticon_1`, `emoticon_2`, and `emoticon_3` for each row. What is the maximum value observed in this new feature across the entire dataset?

In [5]:
train["total_emoticons"] = (
    train["emoticon_1"] + train["emoticon_2"] + train["emoticon_3"]
)

train["total_emoticons"].max()

np.int64(60)

### Question 4

Calculate the median character length (including spaces) of the comment column for all entries where label is equal to 3. (**Note:** Treat any missing comments as empty strings).

In [6]:
train["comment"] = train["comment"].fillna("")
train[train["label"] == 3]["comment"].str.len().median()

np.float64(128.0)

### Question 5

If you apply Min-Max Scaling to the upvote column to normalize it between the range [0, 1], what would be the resulting scaled value for an observation that originally had exactly 10 upvotes?

- [ ] 0.0345
- [ ] 0.0612
- [ ] 0.0521
- [x] 0.0498

In [7]:
scaler = MinMaxScaler()
train["upvote_scaled"] = scaler.fit_transform(train[["upvote"]])

train.loc[train["upvote"] == 10, "upvote_scaled"].iloc[0].round(4)

np.float64(0.0498)

### Question 6

What is the average word count (number of words separated by whitespace) for comments that are assigned a label of 1? (Round your answer to 2 decimal places).

In [8]:
train[train["label"] == 1]["comment"].str.split().str.len().mean().round(2)  # type: ignore[ty:unresolved-attribute]

np.float64(57.23)

### Question 7

How many comments in the entire dataset contain the substring "Trump"? The search should be case-insensitive.

In [9]:
train["comment"].str.contains("Trump", case=False, regex=False).sum()

np.int64(24398)

### Question 8

Take the comment at the very first row (index 0).

- Remove all punctuation.
- Consider the following words as stop words and remove the same :  ['a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to', 'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his', 'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your']

How many words are left in the text after these two operations?

In [10]:
comment = train.loc[0, "comment"]

# fmt: off
stop_words =  [
    'a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to',
    'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have',
    'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his',
    'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your'
]
# fmt: on

len([w for w in re.sub(r"[^\w\s]", "", comment).split() if w.lower() not in stop_words])  # type: ignore[ty:no-matching-overload]

12

### Question 9

Convert all the text in comment column to lowercase and tokenizing using whitespace. Compute the total number of unique tokens in the dataset and enter the value.

In [11]:
train["comment"] = train["comment"].str.lower()
unique_tokens = set()
for comment in train["comment"]:
    unique_tokens.update(comment.split())

len(unique_tokens)

425413

### Question 10

Apply the TfidfVectorizer to the comment column of train.csv with stop_words as "english", min_df as 5 and ngram_range as (1,2).

How many TF-IDF features are generated using this configuration?

In [12]:
tfidf = TfidfVectorizer(stop_words="english", min_df=5, ngram_range=(1, 2))

tfidf.fit(train["comment"])
len(tfidf.get_feature_names_out())

132974

## Milestone 3: Model Improvement

**Deadline:** `Mar 1, 2026`

1. Dimensionality Reduction
2. Feature Selection
3. Retrain the models
    - [Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html)
    - [Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
    - [Support Vector Machines](https://scikit-learn.org/stable/modules/svm.html)

### Import libraries

In [1]:
import os
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

### Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

### Question 1

Split the train dataset using train_test_split with random_state=42 such that 80% is training data and remaining 20% is validation data. store the data as X_train, X_val, y_train, y_val. Lets say training data (X_train) has shape (a,b) where a is number of rows and b is the number of features, similarly validation set (X_val) has shape (c,d) where c is the number of rows and d is the number of features. what will be the value of a + c ?

In [4]:
X = train.drop(columns=["post_id", "label"])
y = train["label"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
X_train.shape[0] + X_val.shape[0]

198000

### Question 2

After converting created_date to a datetime object and extracting the day, month, and year for your X_train and X_val created in the first question, identify the most frequently occurring month across all years in X_train. Which month is it?

In [6]:
X_train["created_date"] = pd.to_datetime(X_train["created_date"])
X_val["created_date"] = pd.to_datetime(X_val["created_date"])

X_train["day"] = X_train["created_date"].dt.day
X_train["month"] = X_train["created_date"].dt.month
X_train["year"] = X_train["created_date"].dt.year

X_val["day"] = X_val["created_date"].dt.day
X_val["month"] = X_val["created_date"].dt.month
X_val["year"] = X_val["created_date"].dt.year

X_train["month"].mode()[0]

np.int32(5)

### Question 3

Impute the null values of categorical features with the value 'none'. Encode only `'religion', 'gender', 'race'` features using one hot encoding, setting `handle_unknown='ignore'`. Make sure the output is a pandas dataframe. Let's say the shape of X_train after imputing is (a,b), what is the value of b?\
**Note:** make sure to transform X_val as well using one hot encoding.

In [7]:
cat_features = X_train.select_dtypes(include="str").columns
cat_features

Index(['race', 'religion', 'gender', 'comment'], dtype='str')

In [8]:
X_train[cat_features] = X_train[cat_features].fillna("none")
X_val[cat_features] = X_val[cat_features].fillna("none")

In [9]:
one_hot = ColumnTransformer(
    [
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["religion", "gender", "race"],
        )
    ],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_train = one_hot.fit_transform(X_train)
X_val = one_hot.transform(X_val)

In [10]:
X_train.shape[1]

32

### Question 4

Apply **CountVectorizer** to the column **comment** of X_train and X_val obtained in previous questions, what is the sum of the counts for the document at index 1 (the second row) of the transformed X_train sparse matrix?\
**Note:** Please ensure to transform X_val as well.

In [11]:
count_vec = CountVectorizer()

X_train_count = count_vec.fit_transform(X_train["comment"])
X_val_count = count_vec.transform(X_val["comment"])

In [12]:
X_train_count[1].sum()

np.int64(41)

### Question 5

Convert **'disability'** feature into integer type, with True being mapped to 1 and False mapped to 0. What is the sum of all the 'disability values in X_train and X_val after the transformation?

In [13]:
X_train["disability"] = X_train["disability"].map({True: 1, False: 0})
X_val["disability"] = X_val["disability"].map({True: 1, False: 0})

In [14]:
X_train["disability"].sum() + X_val["disability"].sum()

np.int64(2743)

### Question 6

Scale the numeric features using **StandardScaler**. What are the number of features seen during fit for X_train? Ensure that you drop all **datetime** columns first.

In [15]:
dt_cols = X_train.select_dtypes(
    include=["datetime64[ns]", "datetime64[ns, UTC]"]
).columns
X_train_no_dt = X_train.drop(columns=dt_cols)

num_features = X_train_no_dt.select_dtypes(include="number").columns
scaler = StandardScaler()
scaler.fit(X_train_no_dt[num_features])

scaler.n_features_in_  # type: ignore

30

### Question 7

Preprocess the data in the following manner and train the model:

Use the full raw data and first split it into training and validation sets so that 80% of the data forms **X_train, y_train** and the remaining 20% forms **X_val, y_val**.

**After splitting:**

1. Impute missing values **by fitting the SimpleImputer on X_train**. You are free to use any valid strategy for Imputation.
2. Use the fitted imputer to transform both **X_train and X_val**
3. Ensure that after imputation no value in X_train or X_val is negative.
4. If any negative value exists, convert it to positive using the absolute value function
5. Convert the created_date feature into a **datetime** object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.
6. Transform the "comment" feature using **TfidfVectorizer** with stop_words='english'. Make sure to transform X_val accordingly.
7. Encode all the categorical features using one hot encoder with handle_unknown='ignore'. Make sure to transform X_val accordingly.
8. Now train a Multinomial Naive Bayes model on the preprocessed data. Use a pipeline if you need to.

What is the macro F1 score obtained on the train dataset?

**Note:** You may train Multinomial Naive Bayes model on the preprocessed data which has been obtained from previous questions and give the answer, both ways the answers will be accepted.

If some feature is creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).

In [16]:
def preprocess(df):
    df = df.copy()
    df["day"] = df["created_date"].dt.day
    df["month"] = df["created_date"].dt.month
    df["year"] = df["created_date"].dt.year
    df.drop(columns=["created_date"], inplace=True)

    num_features = df.select_dtypes(include="number").columns
    df[num_features] = df[num_features].abs()
    return df

In [17]:
X_train_proc = preprocess(X_train)
X_val_proc = preprocess(X_val)

In [18]:
preprocessor = ColumnTransformer(
    transformers=[("tfidf", TfidfVectorizer(stop_words="english"), "comment")],
    remainder="passthrough",
)

pipeline = Pipeline([("preprocessor", preprocessor), ("clf", MultinomialNB())])

pipeline.fit(X_train_proc, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('tfidf', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains 

In [19]:
y_train_pred = pipeline.predict(X_train_proc)
f1_score(y_train, y_train_pred, average="macro")

0.4841311504632059

### Question 8

What is the macro F1 score obtained on the validation dataset?

In [20]:
y_val_pred = pipeline.predict(X_val_proc)
f1_score(y_val, y_val_pred, average="macro")

0.45926776710921624

### Question 9

Process the data in the following manner as given below and train the model.

Use the full raw data and first split it into training and validation sets so that 80% of the data forms X_train, y_train and the remaining 20% forms **X_val, y_val**.

After splitting:
1. Impute missing values **by fitting the SimpleImputer on X_train**. You are free to use any valid strategy for Imputation.
2. Use the fitted imputer to transform both **X_train and X_val**
3. Ensure that after imputation no value in X_train or X_val is negative.
4. If any negative value exists, convert it to positive using the absolute value function
5. Convert the created_date feature into a **datetime** object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.
6. Create a new binary categorical column named **is_weekend** where Saturday and Sunday are represented as **1** and all other days as **0**.
7. Apply **TfidfVectorizer(stop_words='english')** to the comment column. Make sure to transform X_val accordingly.
8. Apply **OneHotEncoder(handle_unknown='ignore')** to the categorical features including newly created categorical feature **is_weekend**. Make sure to transform X_val accordingly.

After all above transformation, train **MultinomialNB** model.

After training, use the model to predict the labels for X_train and calculate the **Macro F1 score**. What is the resulting score rounded to four decimal places?

**Note:** If some features are creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).

In [21]:
def preprocess_v2(df):
    df = df.copy()
    df["day"] = df["created_date"].dt.day
    df["month"] = df["created_date"].dt.month
    df["year"] = df["created_date"].dt.year
    df["is_weekend"] = df["created_date"].dt.dayofweek.apply(
        lambda x: 1 if x >= 5 else 0
    )
    df.drop(columns=["created_date"], inplace=True)

    num_features = df.select_dtypes(include="number").columns
    df[num_features] = df[num_features].abs()
    return df

In [22]:
X_train_proc_v2 = preprocess_v2(X_train)
X_val_proc_v2 = preprocess_v2(X_val)

In [23]:
preprocessor = ColumnTransformer(
    [
        ("tfidf", TfidfVectorizer(stop_words="english"), "comment"),
        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["is_weekend"]),
    ],
    remainder="passthrough",
)

pipeline = Pipeline([("preprocessor", preprocessor), ("clf", MultinomialNB())])

pipeline.fit(X_train_proc_v2, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('tfidf', ...), ('ohe', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [24]:
y_train_pred = pipeline.predict(X_train_proc_v2)
f1_score(y_train, y_train_pred, average="macro")

0.4840696062885761

### Question 10

What is the macro F1-score on validation set for the previous question's setup?

In [25]:
y_val_pred = pipeline.predict(X_val_proc_v2)
f1_score(y_val, y_val_pred, average="macro")

0.4592333361559341

## Milestone 4: Ensemble Learning and Multi-layer Perceptron

**Deadline:** `Mar 9, 2026`

1. [Ensemble](https://scikit-learn.org/stable/modules/ensemble.html) (Bagging, Boosting, Stacking)
2. [Multi-layer Perceptron](https://scikit-learn.org/stable/modules/neural_networks_supervised.html#multi-layer-perceptron)

### Import libraries

In [40]:
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

### Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

### Question 1

Load the dataset and fill the NaN values in comment column with empty string then follow the below steps:

1. Split the dataset after doing above step using train_test_split into 60% train and 40% validation. Keep random_state = 2306 and set stratify to the target variable (y).

    After splitting, compute the class-wise counts of the training labels and validation labels using value_counts().

    Before storing the counts, sort the class labels in ascending order to ensure consistent ordering. Store the resulting counts as NumPy arrays.

2. Load the same dataset and split it again using train_test_split, keeping all configurations the same as in Part A (60% train and 40% validation, random_state = 2306), but set stratify = None.

    Again, compute the sorted class-wise counts of the training labels and validation labels and store them as NumPy arrays.

3. **Validation Distribution Difference**

    Using the validation label counts from Part A (stratified) and Part B (non-stratified)
    Convert both validation class count arrays into **proportions** by dividing each count by the total number of validation samples.

    Make sure the class labels remain in ascending order (same order as before).

For each class, compute the absolute difference between the two proportions.

Find the maximum of these absolute differences.

In [4]:
train["comment"] = train["comment"].fillna("")
test["comment"] = test["comment"].fillna("")

X = train.drop(columns=["label"])
y = train["label"]

In [5]:
X_train_A, X_val_A, y_train_A, y_val_A = train_test_split(
    X, y, test_size=0.4, random_state=2306, stratify=y
)

In [6]:
train_counts_A = y_train_A.value_counts().sort_index().to_numpy()
val_counts_A = y_val_A.value_counts().sort_index().to_numpy()

In [7]:
X_train_B, X_val_B, y_train_B, y_val_B = train_test_split(
    X, y, train_size=0.6, test_size=0.4, random_state=2306, stratify=None
)

In [8]:
train_counts_B = y_train_B.value_counts().sort_index().to_numpy()
val_counts_B = y_val_B.value_counts().sort_index().to_numpy()

In [9]:
val_prop_A = val_counts_A / val_counts_A.sum()
val_prop_B = val_counts_B / val_counts_B.sum()

np.abs(val_prop_A - val_prop_B).max()

np.float64(0.00060606060606061)

### Question 2

Use the **stratified train–validation split** obtained in Q1 (60% train, 40% validation, random_state=2306, stratify=y).

All preprocessing objects must be **fit only on the training data** and then used to transform both training and validation sets.

1. **Column Removal** \
    From both x_train and x_test: \
    Drop the column "created_date".

2. **Separate Text Column** \
    Before any tabular preprocessing, Extract the "comment" column:
    ```python
    text_x_train = x_train["comment"]
    text_x_test = x_test["comment"]
    ```
    Remove "comment" from both x_train and x_test.

3. **Tabular Preprocessing** \
    Apply a ColumnTransformer with the following exact configuration

    **_Handling categorical columns_** \
    Categorical columns are `["race", "religion", "gender", "disability"]`

    Process the above categorical columns using below techniques:
    ```python
    SimpleImputer(strategy="most_frequent")
    OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    ```

    **_Handling numerical columns_** \
    numerical columns are `["post_id", "emoticon_1", "emoticon_2", "emoticon_3", "upvote", "downvote", "if_1", "if_2"]` \
    Process the above numerical columns using below techniques:
    ```python
    SimpleImputer(strategy="mean")
    StandardScaler() (default parameters)
    ```

    **_ColumnTransformer Settings_** \
    `remainder="passthrough"` \
    Fit only on `x_train` \
    Transform both `x_train` and `x_test` \
    Store the transformed matrices as:
    ```
    x_train_tabular
    x_test_tabular
    ```


4. **Text Cleaning**
    Define the following function exactly:

    ```python
    def normalize_text(text):
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\s+', '  ', text).strip()
        return text
    ```

    apply it using .apply() on both text_x_train and text_x_test then store the result as text_x_train_norm, text_x_test_norm

5. **TF-IDF Vectorization**
    **Use:** `TfidfVectorizer(stop_words="english", max_features=5000)`

    Fit only on text_x_train_norm and Transform both train and test text
    Store as: `tf_idf_train`, `tf_idf_test`
    Do not change any other parameters.

6. **Combine Features**

    **Use:** hstack to combine features in the following exact order:

    ```
    X_train_final = hstack([x_train_tabular, tf_idf_train])
    X_test_final = hstack([x_test_tabular, tf_idf_test])
    ```
    The result must remain a sparse matrix.

    Give the sum of all values of X_train_final, enter the answer upto 3 decimal places.

**Note:** handle the null values of comment column (if any) by replacing them with empty string to avoid errors.

In [10]:
X_train = X_train_A.copy()
X_val = X_val_A.copy()

y_train = y_train_A.copy()
y_val = y_val_A.copy()

In [11]:
X_train["comment"] = X_train["comment"].fillna("")
X_val["comment"] = X_val["comment"].fillna("")

In [12]:
text_x_train = X_train["comment"]
text_x_test = X_val["comment"]

In [13]:
X_train.drop(columns=["created_date", "comment"], inplace=True, errors="ignore")
X_val.drop(columns=["created_date", "comment"], inplace=True, errors="ignore")

In [14]:
cat_features = ["race", "religion", "gender", "disability"]
num_features = [
    "post_id",
    "emoticon_1",
    "emoticon_2",
    "emoticon_3",
    "upvote",
    "downvote",
    "if_1",
    "if_2",
]

if not IS_KAGGLE:
    X_train[cat_features] = X_train[cat_features].astype(object)
    X_val[cat_features] = X[cat_features].astype(object)

In [15]:
transformer = ColumnTransformer(
    [
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "encoder",
                        OneHotEncoder(handle_unknown="ignore", sparse_output=True),
                    ),
                ]
            ),
            cat_features,
        ),
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="mean")),
                    ("scaler", StandardScaler()),
                ]
            ),
            num_features,
        ),
    ],
    remainder="passthrough",
)
transformer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``.

In [16]:
x_train_tabular = transformer.fit_transform(X_train)
x_test_tabular = transformer.transform(X_val)

In [17]:
def normalize_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\s+", "  ", text).strip()
    return text

In [18]:
text_x_train_norm = text_x_train.apply(normalize_text)
text_x_test_norm = text_x_test.apply(normalize_text)

In [19]:
tf_idf = TfidfVectorizer(stop_words="english", max_features=5000)
tf_idf

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.

In [20]:
tf_idf_train = tf_idf.fit_transform(text_x_train_norm)
tf_idf_test = tf_idf.transform(text_x_test_norm)

In [21]:
X_train_final = hstack([x_train_tabular, tf_idf_train])
X_test_final = hstack([x_test_tabular, tf_idf_test])

In [22]:
X_train_final.sum().round(3)

np.float64(904262.933)

### Instructions for further questions

This section is not a question, its the instruction for further questions
For all subsequent questions (Q3 onwards), reduce the feature space using TruncatedSVD(n_components=300, random_state=2306)

**Instructions:**
- Fit the TruncatedSVD **only on X_train_final**.
- Transform both  X_train_final and X_test_final
- Store the transformed matrices as X_train_reduced and X_test_reduced.

Use X_train_reduced for all model training in the remaining questions. Do not refit SVD for each question. Do not change n_components.

- [x] This is just instruction and not the question, check the box once you read it carefully and prepare X_train_reduced, X_test_reduced.

In [23]:
svd = TruncatedSVD(n_components=300, random_state=2306)
svd

,"n_components n_components: int, default=2Desired dimensionality of output data.If algorithm='arpack', must be strictly less than the number of features.If algorithm='randomized', must be less than or equal to the number of features.The default value is useful for visualisation. For LSA, a value of100 is recommended.",300
,"algorithm algorithm: {'arpack', 'randomized'}, default='randomized'SVD solver to use. Either ""arpack"" for the ARPACK wrapper in SciPy(scipy.sparse.linalg.svds), or ""randomized"" for the randomizedalgorithm due to Halko (2009).",'randomized'
,"n_iter n_iter: int, default=5Number of iterations for randomized SVD solver. Not used by ARPACK. Thedefault is larger than the default in:func:`~sklearn.utils.extmath.randomized_svd` to handle sparsematrices that may have large slowly decaying spectrum.",5
,"n_oversamples n_oversamples: int, default=10Number of oversamples for randomized SVD solver. Not used by ARPACK.See :func:`~sklearn.utils.extmath.randomized_svd` for a completedescription... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD solver.Not used by ARPACK. See :func:`~sklearn.utils.extmath.randomized_svd`for more details... versionadded:: 1.1",'auto'
,"random_state random_state: int, RandomState instance or None, default=NoneUsed during randomized svd. Pass an int for reproducible results acrossmultiple function calls.See :term:`Glossary `.",2306
,"tol tol: float, default=0.0Tolerance for ARPACK. 0 means machine precision. Ignored by randomizedSVD solver.",0.0


In [24]:
X_train_reduced = svd.fit_transform(X_train_final)
X_test_reduced = svd.transform(X_test_final)

#### Question 3

Use the preprocessed training data X_train_reduced.

Perform hyperparameter tuning for a RandomForestClassifier using RandomizedSearchCV.

**Model Setup:**

Initialize the model as:
- RandomForestClassifier(random_state=2306)
- Do not manually set n_estimators or max_depth here, as they will be tuned.
- Search only over the following parameter values:
    ```python
    n_estimators: [50, 100, 200]
    max_depth: [5, 10, 15]
    ```

Do not include any other hyperparameters in the search.

RandomizedSearchCV Configuration is given below
```python
n_iter = 5
cv = 3
random_state = 2306
n_jobs = -1
```

Do not manually specify the scoring parameter (use default). \
After running: `randomized_search.fit(X_train_reduced, y_train)`

Retrieve the best hyperparameters using: `randomized_search.best_params_`

Return the best value of **n_estimators** selected by RandomizedSearchCV.

Return only the integer value.

In [25]:
model = RandomForestClassifier(random_state=2306)
model

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [26]:
random_cv = RandomizedSearchCV(
    estimator=model,
    param_distributions={
        "n_estimators": [50, 100, 200],
        "max_depth": [5, 10, 15],
    },
    n_iter=5,
    cv=3,
    random_state=2306,
    n_jobs=-1,
)
random_cv

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...om_state=2306)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [5, 10, ...], 'n_estimators': [50, 100, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5

In [27]:
random_cv.fit(X_train_reduced, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...om_state=2306)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [5, 10, ...], 'n_estimators': [50, 100, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5

In [28]:
random_cv.best_params_["n_estimators"]

200

#### Question 4

Use the same preprocessed training data (`X_train_reduced`, `y_train`).
Do not modify or recompute preprocessing.

1. **Train AdaBoost** \
    Train an AdaBoostClassifier with the following exact parameters:
    ```python
    n_estimators = 50
    random_state = 2306
    ```

    Use all other parameters as default. \
    **Note:**
    - Do not manually set the algorithm parameter.
    - Use the default behavior of AdaBoostClassifier.

    Fit the model on: `X_train_reduced` , `y_train`

2. **Extract Estimator Errors** \
    After fitting, access: `model.estimator_errors_` \
    This returns a NumPy array of length 50 containing the error of each weak learner.

3. **Compute Variance** \
    Compute the variance of the 50 estimator errors using: `np.var(model.estimator_errors_)` \
    **Important:**
    - Use NumPy’s default variance computation.

4. Final Answer
Return the variance rounded to 4 decimal places.

In [32]:
ada = AdaBoostClassifier(n_estimators=50, random_state=2306)
ada

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",50
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",2306


In [33]:
ada.fit(X_train_reduced, y_train)

,"estimator estimator: object, default=NoneThe base estimator from which the boosted ensemble is built.Support for sample weighting is required, as well as proper``classes_`` and ``n_classes_`` attributes. If ``None``, thenthe base estimator is :class:`~sklearn.tree.DecisionTreeClassifier`initialized with `max_depth=1`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",None
,"n_estimators n_estimators: int, default=50The maximum number of estimators at which boosting is terminated.In case of perfect fit, the learning procedure is stopped early.Values must be in the range `[1, inf)`.",50
,"learning_rate learning_rate: float, default=1.0Weight applied to each classifier at each boosting iteration. A higherlearning rate increases the contribution of each classifier. There isa trade-off between the `learning_rate` and `n_estimators` parameters.Values must be in the range `(0.0, inf)`.",1.0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given at each `estimator` at eachboosting iteration.Thus, it is only used when `estimator` exposes a `random_state`.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",2306


In [35]:
np.var(ada.estimator_errors_).round(4)

np.float64(0.0058)

#### Question 5

Train a `RandomForestClassifier(n_estimators=100, max_depth=10, random_state=2306)` using the preprocessed training data `(X_train_reduced, y_train)`.

**After fitting:**
- Extract feature_importances_.
- Give the index of the feature with the maximum importance value.

**Note:** If multiple features share the exact same maximum importance, return the smallest index.

In [36]:
rfc = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=2306)
rfc

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [37]:
rfc.fit(X_train_reduced, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [38]:
rfc.feature_importances_.argmax()

np.int64(4)

#### Question 6

Initialize an MLPClassifier with:

```python
hidden_layer_sizes = (128, 64, 32)
activation = "relu"
random_state = 2306
```

Use the preprocessed training data `(X_train_reduced , y_train)`.
Let N be the number of input features (i.e., `X_train_reduced.shape[1]`).

Compute the total number of weights (excluding biases) in the network.
Only count connection weights between layers. Do not include bias terms.

The network structure is:
- Input layer: N neurons
- Hidden Layer 1: 128 neurons
- Hidden Layer 2: 64 neurons
- Hidden Layer 3: 32 neurons
- Output Layer: 4 neurons

Use your actual value of N from `X_train_reduced`.
Return the final integer count of weights.

In [41]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32), activation="relu", random_state=2306
)
mlp

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(128, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",2306


In [42]:
mlp.fit(X_train_reduced, y_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(128, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",200
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",2306


In [50]:
sum(w.size for w in mlp.coefs_)

48768

#### Question 7

Train the same MLP architecture defined in Q6 using the preprocessed training data `(X_train_reduced, y_train)` with the following additional parameters:

`solver = "adam", max_iter = 5, batch_size = 32, random_state = 2306`

Do not modify any other parameters (keep all others as default).
After training completes (i.e., after 5 iterations), access: `model.loss_`
This returns the final value of the loss function after the last iteration.
Return the value of `model.loss_` rounded to 4 decimal places.

In [44]:
mlp_v2 = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    max_iter=5,
    batch_size=32,
    random_state=2306,
)
mlp_v2

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(128, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",32
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",5
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",2306


In [45]:
mlp_v2.fit(X_train_reduced, y_train)

V:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(128, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.0001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",32
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",5
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",2306


In [46]:
round(mlp_v2.loss_, 4)

np.float64(0.3271)

#### Question 8

Train two MLPs using the preprocessed training data `(X_train_reduced, y_train)` with `hidden_layer_sizes=(100,), max_iter=5, and random_state=2306`. use default values for other parameters unless specified.
- Model A: `alpha=0.0001` (Default)
- Model B: `alpha=1.0` (Strong Regularization)

Calculate the Macro F1 Score for both on the training set. What is the absolute difference between both the scores? (answer upto 4 decimal places).

In [47]:
mlp_a = MLPClassifier(
    hidden_layer_sizes=(100,), max_iter=5, alpha=0.0001, random_state=2306
)
mlp_b = MLPClassifier(
    hidden_layer_sizes=(100,), max_iter=5, alpha=1.0, random_state=2306
)

In [48]:
mlp_a.fit(X_train_reduced, y_train)
mlp_b.fit(X_train_reduced, y_train)

V:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(
V:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (5) reached and the optimization hasn't converged yet.
  warnings.warn(


,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(100,)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",1.0
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",5
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",2306


In [51]:
f1_s_a = f1_score(y_train, mlp_a.predict(X_train_reduced), average="macro")
f1_s_b = f1_score(y_train, mlp_b.predict(X_train_reduced), average="macro")

round(abs(f1_s_a - f1_s_b), 4)

0.1327

#### Question 9

Training the MLP model as per Q7, generate a confusion matrix for the validation set. What is the sum of the off-diagonal elements (representing the total number of misclassified instances) divided by the total number of samples in the validation set? (Round to 4 decimal places).

In [52]:
y_val_pred = mlp_v2.predict(X_test_reduced)

In [53]:
cm = confusion_matrix(y_val, y_val_pred)
misclassified = cm.sum() - np.trace(cm)
total_samples = cm.sum()
(misclassified / total_samples).round(4)

np.float64(0.1126)

## Milestone 5: ML End-to-End with Insights

**Deadline:** `Mar 15, 2026`

1. Repeat the ML end-to-end process again with insights into the dataset.

### Import libraries

In [1]:
import os
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import OneHotEncoder

### Config

In [2]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("../datasets/")

### Load data

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

### Instructions for further questions

You are given two datasets:

* **Training dataset:** `df` (full training data)
* **Test dataset:** `test_df` (test data)

> **Note:** We are not going to use `test_df` anywhere in the below questions. However, in the project you will be required to process the test data as well. Due to this reason, we have given the task to perform preprocessing on the test data provided to you in the competition.

The **target variable** for the training dataset (`df`) is **`y`**.
Extract it from the training data and store it as `y`, then **drop the target column from the training data**.

Your task is to **preprocess the data so that it can be used for model training.**

---

#### Step 1: Datetime Feature Engineering

For both **`df` and `test_df`**:

1. Convert the feature **`created_date`** to datetime format using:

   ```
   pd.to_datetime()
   ```

2. Extract the following features from `created_date`:

   * `year`
   * `month`
   * `hour`

3. Add these as **new features**.

4. After extracting these features, **drop the original `created_date` feature**.

---

#### Step 2: Handle Missing Values

Perform the following preprocessing for **both datasets (`df` and `test_df`)**.

##### Categorical Features

Fill missing values with **"Unknown"** for the following features:

* `race`
* `religion`
* `gender`

##### Text Feature

Replace missing values in the **`comment`** feature with an **empty string `""`**.

##### Boolean Feature

Convert the **`disability`** feature to **integer type** using:

```
.astype(int)
```

---

#### Step 3: Feature Transformation using `ColumnTransformer`

Use **`ColumnTransformer`** to process the features as follows.

##### Text Feature

Apply:

```
TfidfVectorizer(stop_words="english", max_features=5000)
```

to the feature:

```
comment
```

---

##### Categorical Features

Apply:

```
OneHotEncoder(drop="first", handle_unknown="ignore")
```

to the following features:

```
["race", "religion", "gender"]
```

---

##### Remaining Features

All remaining features should be passed through unchanged using:

```
remainder="passthrough"
```

This will keep numeric features such as:

* `year`
* `month`
* `hour`
* `disability`
* other numeric features

---

#### Step 4: Transform the Data

Fit the `ColumnTransformer` **only on the training dataset (`df`)**.

Transform both datasets and store the results as:

```
X = preprocessor.fit_transform(df)
X_test = preprocessor.transform(test_df)
```

---

#### Step 5: Create Train–Validation Split

Split the training features and labels `(X, y)` into **training and validation sets** using:

```
train_test_split
```

with the following parameters:

```
test_size = 0.2
random_state = 42
```

Store the outputs as:

```
X_train, X_val, y_train, y_val
```

---

#### Final Output

After completing all steps, you should have the following variables ready for **model training**:

```
X_train
X_val
y_train
y_val
X_test
```

Use this **processed data** for all the questions below.

---

- [x] I have read and performed above preprocessing steps.

In [4]:
X = train.drop(columns=["label"])
y = train["label"]

In [5]:
X["created_date"] = pd.to_datetime(X["created_date"])

X["year"] = X["created_date"].dt.year
X["month"] = X["created_date"].dt.month
X["day"] = X["created_date"].dt.day

X.drop(columns=["created_date"], inplace=True, errors="ignore")

In [6]:
cols = ["race", "religion", "gender"]
X[cols] = X[cols].fillna("Unknown")

X["comment"] = X["comment"].fillna("")
X["disability"] = X["disability"].astype(int)

In [7]:
preprocessor = ColumnTransformer(
    [
        (
            "comment",
            TfidfVectorizer(stop_words="english", max_features=5000),
            "comment",
        ),
        ("cat_features", OneHotEncoder(drop="first", handle_unknown="ignore"), cols),
    ],
    remainder="passthrough",
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('comment', ...), ('cat_features', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fe

In [8]:
X = preprocessor.fit_transform(X)

In [9]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#### Question 1

After applying TF-IDF vectorization on the comment column as per above instructions, what is the maximum number of text features that can be generated?

In [10]:
preprocessor.named_transformers_["comment"].max_features

5000

#### Question 2

What are the number of features in `X_train`?

In [11]:
X_train.shape[1]

5031

#### Question 3

What are the number of features in `X_val`?

In [12]:
X_val.shape[1]

5031

#### Question 4

Train a **Multinomial Naive Bayes** model using the preprocessed data `X_train`, `y_train`. Generate predictions on the **validation dataset (X_val,y_val)** and evaluate the model using **accuracy score**. What **validation accuracy** (using X_val, y_val) does the Naive Bayes model achieve?

(Round the answer to 3 decimal places.)

In [13]:
naive = MultinomialNB()
naive.fit(X_train, y_train)

pred = naive.predict(X_val)
acc = accuracy_score(y_val, pred)
print(f"Validation Accuracy: {acc:.3f}")

Validation Accuracy: 0.807


#### Question 5

What is the precision score for the label 3 from the classification report after performing the Multinomial Naive Bayes on validation set (X_val, y_val)?

(Round the answer to 2 decimal places.)

In [14]:
clf_report = classification_report(y_val, pred)
print(clf_report)

              precision    recall  f1-score   support

           0       0.90      0.89      0.89     22534
           1       0.45      0.81      0.58      3219
           2       0.83      0.71      0.76     12738
           3       0.51      0.27      0.35      1109

    accuracy                           0.81     39600
   macro avg       0.67      0.67      0.65     39600
weighted avg       0.83      0.81      0.81     39600



#### Question 6

Train a **Logistic Regression** model using the training dataset with the following parameters:
```python
max_iter = 500, random_state = 42
```

Generate predictions on the **validation dataset** (X_val, y_val) and compute the **accuracy score**.
What **validation accuracy** does the Logistic Regression model achieve?

(Round the answer to 3 decimal places.)

In [15]:
log_reg = LogisticRegression(max_iter=500, random_state=42)
log_reg.fit(X_train, y_train)

pred_log = log_reg.predict(X_val)
acc_log = accuracy_score(y_val, pred_log)
print(f"Validation Accuracy: {acc_log:.3f}")

Validation Accuracy: 0.838


V:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


#### Question 7

What **training accuracy** does the Logistic Regression model achieve? use X_train, y_train to calculate it.

(Round the answer to 3 decimal places.)

In [16]:
train_pred_log = log_reg.predict(X_train)
acc_log = accuracy_score(y_train, train_pred_log)
print(f"Validation Accuracy: {acc_log:.3f}")

Validation Accuracy: 0.841


#### Question 8

What is the precision score for the label 1 from the classification report after performing the Logistic regression on  validation set (use X_val, y_val)?

(Round the answer to 2 decimal places.)

In [17]:
clf_report_log = classification_report(y_val, pred_log)
print(clf_report_log)

V:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
V:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
V:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisio

              precision    recall  f1-score   support

           0       0.96      0.95      0.95     22534
           1       0.58      0.00      0.00      3219
           2       0.69      0.93      0.79     12738
           3       0.00      0.00      0.00      1109

    accuracy                           0.84     39600
   macro avg       0.56      0.47      0.44     39600
weighted avg       0.81      0.84      0.80     39600



#### Question 9

Perform **hyperparameter tuning** on the Logistic Regression model using **GridSearchCV** with **3-fold cross-validation**. Tune the parameter C using the values **0.1, 1, and 10**, and use the **liblinear solver**. Select the best model from GridSearch and evaluate it on the validation dataset.

Which value of **C out of 0.1, 1, 10** produces the best model?

In [18]:
grid_cv = GridSearchCV(
    estimator=LogisticRegression(solver="lbfgs"),
    param_grid={
        "C": [0.1, 1, 10],
    },
    cv=3,
    n_jobs=-1,
)
grid_cv.fit(X_train, y_train)

V:\Codes\IITM-BS\MLP_Project\Comment Category Prediction Challenge\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fold and candidate parameter indexes are al

In [19]:
best_model = grid_cv.best_estimator_

In [20]:
grid_cv.best_params_["C"]

10

#### Question 10

What **validation accuracy** do you obtain after hyperparameter tuning?

(Round the answer to 2 decimal places.)

In [21]:
best_pred = best_model.predict(X_val)
acc_best = accuracy_score(y_val, best_pred)
print(f"Validation Accuracy: {acc_best:.3f}")

Validation Accuracy: 0.838
